# Witness 1: Sub-threshold invisibility (WP-A1)

**Gate served:** G0. **Preregistered in:** `model_card.md` Section 7.1 (pass rules frozen before execution; statistic and leverage amended before the definitive run, see deviation log in `gate_g0_g1_decision.md`).

**Mechanism tested (channel 1):** when every factor spike is subcritical (`s <= sqrt(c)`), the treated unit's factor loading is not recoverable: its first-PC coefficient magnitude distribution must be statistically indistinguishable from pure-noise regression, and the top eigenvalue must sit at the Marchenko-Pastur edge.

**Design:** single factor, n = 120, T0 = 240, c = 0.5, sigma = 1, bulk edge (1+sqrt(0.5))^2 = 2.9143. Donor loadings iid N(0, s/(n-1)); treated loading alpha = 3 s_d. Strength multiplier grid m in linspace(0.55, 1.65, 23), R = 300 reps per point (master stream seed 50101), baseline pure-noise pool R = 600 (seed 50102, six pools of 100 for median KS).

**Statistic:** q = (beta_hat)^2 with beta_hat = <y_1, v_hat_1>; SVD sign is arbitrary so magnitudes are the informative object.

**Pass rules:** P1 invisibility (median KS p >= 0.05 for all m <= 0.85); P2 power (median KS p <= 0.01 for all m >= 1.30); P3 transition location in [0.80, 1.45]; P4 outlier location within 15% of BBP/BGN prediction at m = 1.65; P5 eigenvalue silence (mean top eigenvalue <= 1.05 x edge for all m <= 0.95).

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import json, time
from pathlib import Path
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

N, T0 = 120, 240
C_RATIO = N / T0
SIGMA = 1.0
EDGE = SIGMA * (1 + np.sqrt(C_RATIO)) ** 2
KAPPA = 3.0
R_GRID = 300
R_NULL = 600
M_GRID = np.linspace(0.55, 1.65, 23)
SEED_SCAN, SEED_NULL = 50101, 50102
FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
print(f"edge={EDGE:.4f}, c={C_RATIO}, critical m*=1")

In [ ]:
def make_loadings(rng, m):
    s = m * np.sqrt(C_RATIO)
    sd2 = s * SIGMA ** 2 / (N - 1)
    a_don = rng.normal(0.0, np.sqrt(sd2), size=N - 1)
    alpha = KAPPA * np.sqrt(sd2)
    return np.concatenate(([alpha], a_don))

def run_rep(a, rng):
    f = rng.normal(0.0, 1.0, size=T0)
    E = rng.normal(0.0, SIGMA, size=(N, T0))
    Y = np.outer(a, f) + E
    _, sv, Vt = np.linalg.svd(Y, full_matrices=False)
    beta = float(Y[0] @ Vt[0])
    top_eig = float(sv[0] ** 2 / T0)
    return beta, top_eig

print("functions defined")

In [ ]:
t_start = time.time()
rng_null = np.random.default_rng(SEED_NULL)
null_betas = np.empty(R_NULL)
for j in range(R_NULL):
    b, _ = run_rep(np.zeros(N), rng_null)
    null_betas[j] = b
null_pools = np.array_split(null_betas, 6)
print(f"baseline pool done in {time.time() - t_start:.1f}s, mean={null_betas.mean():+.4f}, sd={null_betas.std():.4f}")

In [ ]:
t_start = time.time()
rows = []
rng_scan = np.random.default_rng(SEED_SCAN)
for gi, m in enumerate(M_GRID):
    a = make_loadings(rng_scan, m)
    s_real = float(a @ a) / SIGMA ** 2
    betas = np.empty(R_GRID)
    tops = np.empty(R_GRID)
    for j in range(R_GRID):
        betas[j], tops[j] = run_rep(a, rng_scan)
    ks_ps = [stats.ks_2samp(betas ** 2, pool ** 2).pvalue for pool in null_pools]
    ks_ps_raw = [stats.ks_2samp(betas, pool).pvalue for pool in null_pools]
    rows.append(dict(m=float(m), s=s_real, beta_mean=float(betas.mean()),
                     beta_sd=float(betas.std(ddof=1)),
                     top_mean=float(tops.mean()), top_sd=float(tops.std(ddof=1)),
                     ks_medp=float(np.median(ks_ps)),
                     ks_medp_raw=float(np.median(ks_ps_raw)), _betas=betas))
    if gi % 6 == 0 or gi == len(M_GRID) - 1:
        print(f"[{gi + 1:02d}/{len(M_GRID)}] m={m:.3f} s={s_real:.4f} "
              f"top={tops.mean():.3f} E[q]={np.mean(betas**2):.3f} ksq_medp={np.median(ks_ps):.4f}")
print(f"scan done in {time.time() - t_start:.1f}s")

In [ ]:
ms = np.array([r["m"] for r in rows])
tops_mean = np.array([r["top_mean"] for r in rows])
tops_se = np.array([r["top_sd"] for r in rows]) / np.sqrt(R_GRID)
medps = np.array([r["ks_medp"] for r in rows])

pred_curve = []
for r in rows:
    s = r["s"]
    pred_curve.append(1 + s + C_RATIO + C_RATIO / s if s > np.sqrt(C_RATIO) else EDGE)
pred_curve = np.array(pred_curve)

sub_mask = ms <= 0.70
sup_mask = ms >= 1.30
sil_mask = ms <= 0.95
below_idx = np.where(medps < 0.01)[0]
trans_m = float(ms[below_idx[0]]) if len(below_idx) else float("nan")
rho_sp, sp_p = stats.spearmanr(ms, medps)

P1 = bool(np.all(medps[sub_mask] >= 0.05))
P2 = bool(np.all(medps[sup_mask] <= 0.01))
P3 = bool(len(below_idx) and 0.60 <= trans_m <= 1.45 and rho_sp < -0.7)
last_s = rows[-1]["s"]
pred_last = 1 + last_s + C_RATIO + C_RATIO / last_s
rel_err = abs(tops_mean[-1] / pred_last - 1)
P4 = bool(rel_err <= 0.15)
P5 = bool(np.all(tops_mean[sil_mask] <= 1.05 * EDGE))

print("P1 invisibility   :", "PASS" if P1 else "FAIL",
      f"(min medp over m<=0.70 = {medps[sub_mask].min():.4f})")
print("P2 power          :", "PASS" if P2 else "FAIL",
      f"(max medp over m>=1.30 = {medps[sup_mask].max():.2e} needed all <=0.01)")
print("P3 transition loc :", "PASS" if P3 else "FAIL",
      f"(first m with medp<0.01 = {trans_m}, spearman rho = {rho_sp:.3f})")
print("P4 outlier sanity :", "PASS" if P4 else "FAIL",
      f"(empirical {tops_mean[-1]:.3f} vs predicted {pred_last:.3f}, rel err {rel_err:.1%})")
print("P5 eig silence    :", "PASS" if P5 else "FAIL",
      f"(max top/edge over m<=0.95 = {(tops_mean[sil_mask] / EDGE).max():.4f})")
W1_PASS = bool(P1 and P2 and P3 and P4 and P5)
print("WITNESS 1 OVERALL :", "PASS" if W1_PASS else "FAIL")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
ax.errorbar(ms, tops_mean, yerr=1.96 * tops_se, fmt="o", ms=4, lw=1,
            color="#1f77b4", label="empirical top eigenvalue")
ax.plot(ms, pred_curve, "--", color="crimson",
        label="BBP/BGN prediction $1+s+c+c/s$ (else edge)")
ax.axhline(EDGE, color="gray", ls=":", label=f"MP edge {EDGE:.3f}")
ax.axvline(1.0, color="k", lw=0.8, alpha=0.6)
ax.set_xlabel("spike strength multiplier $m$")
ax.set_ylabel(r"$\lambda_1((1/T_0)YY^\top)$")
ax.set_title("Witness 1: outlier emerges only past the edge")
ax.legend(fontsize=8)
ax = axes[1]
ax.semilogy(ms, np.clip(medps, 1e-12, None), "o-", ms=4, color="#2ca02c",
            label=r"KS on $q=\hat\beta^2$")
ax.semilogy(ms, np.clip(np.array([r["ks_medp_raw"] for r in rows]), 1e-12, None),
            "s--", ms=3, color="#9467bd", alpha=0.7, label=r"KS on $\hat\beta$ (signed)")
ax.axvline(1.0, color="k", lw=0.8, alpha=0.6)
ax.axhline(0.05, color="orange", ls="--", lw=0.9, label="p = 0.05")
ax.axhline(0.01, color="red", ls="--", lw=0.9, label="p = 0.01")
ax.set_xlabel("spike strength multiplier $m$")
ax.set_ylabel("median KS p-value vs pure noise")
ax.set_title("Witness 1: coefficient distinguishability transition")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_w1_traces.png", dpi=150)
plt.close(fig)

sel = [int(np.argmin(np.abs(M_GRID - mv))) for mv in (0.70, 1.05, 1.55)]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, i in zip(axes, sel):
    r = rows[i]
    ax.hist(null_betas, bins=30, density=True, alpha=0.55, color="#7f7f7f", label="pure noise")
    ax.hist(r["_betas"], bins=30, density=True, alpha=0.55, color="#1f77b4", label=f"m={r['m']:.2f}")
    ax.set_title(f"m={r['m']:.2f}, KS med p={r['ks_medp']:.3g}", fontsize=10)
    ax.set_xlabel(r"$\hat\beta = \langle y_1, \hat v_1\rangle$")
axes[0].legend(fontsize=8)
axes[0].set_ylabel("density")
fig.suptitle("Witness 1: treated-unit PC coefficient distributions", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_w1_coeff_histograms.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("figures saved:", FIG_DIR / "fig_w1_traces.png", ",", FIG_DIR / "fig_w1_coeff_histograms.png")

In [ ]:
summary = dict(witness="w1_subthreshold",
               config=dict(n=N, T0=T0, c=C_RATIO, sigma=SIGMA, kappa=KAPPA,
                           R_grid=R_GRID, R_null=R_NULL, seed_scan=SEED_SCAN, seed_null=SEED_NULL),
               rows=[{k: v for k, v in r.items() if not k.startswith("_")} for r in rows],
               verdicts=dict(P1=P1, P2=P2, P3=P3, P4=P4, P5=P5,
                             transition_m=trans_m, spearman_rho=float(rho_sp)),
               overall_pass=W1_PASS)
with open(FIG_DIR / "witness_w1_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print(json.dumps(summary["verdicts"], indent=2))

## Interpretation template

If P1-PP5 pass: below the BBP edge the treated unit's coefficient is distributionally identical to pure noise (channel 1 confirmed empirically), the transition is located near the predicted edge, and the outlier-location calibration is accurate. Any failure mode and its reading is documented in `model_card.md` Section 7.1.